# Ganglion demo · EMG 손동작 → 방향키

OpenBCI Ganglion 근전도로 손동작 4개를 분류하고, 결과를 **방향키 입력**으로 내보낸다.
`Ganglion_Tutorial_2_Classification` 에서 연결·분류·키출력만 남긴 축약판.

---

## 처리 파이프라인

```
Ganglion 4채널 @ 200 Hz
        │
        │  ① 에포킹   최근 EPOCH_SEC 초를 잘라낸다  →  (샘플, 4채널)
        ▼
   ┌─────────────────────────────────────────┐
   │ ② DC 제거   REMOVE_DC                    │  기본 켬
   ├─────────────────────────────────────────┤
   │ ③ 노치      USE_NOTCH                    │  전원 잡음 제거
   │    IIR biquad · filtfilt (영위상)         │  ← 현재 기본 끔
   ├─────────────────────────────────────────┤
   │ ④ 대역통과  USE_BANDPASS                 │  EMG 성분만 남김
   │    Butterworth FILTER_ORDER차 · filtfilt  │  ← 현재 기본 끔
   └─────────────────────────────────────────┘
        │
        │  ⑤ 포락선   |Hilbert(x)|  →  진동을 "세기 윤곽"으로
        │  ⑥ 평균     에포크 구간 평균  →  채널당 숫자 1개
        ▼
   특징 벡터 (4차원)  [Ch1 활성, Ch2 활성, Ch3 활성, Ch4 활성]
        │
        │  ⑥ Random Forest (트리 N_TREES개)
        ▼
   순간 예측 1~4
        │
        │  ⑦ 최근 VOTE_LEN 회 다수결  →  튀는 예측 흡수
        ▼
   안정 판정  →  ⑧ keydown 유지 (판정 바뀔 때만, 바뀐 키만 교체)
```

동작별 키 배치 (`KEY_COMBO` 에서 변경):

| 동작 | 키 | 비고 |
| --- | --- | --- |
| 1 Reverse | `DOWN` | **UP 없음** — 후진 |
| 2 Left | `LEFT` + `UP` | 좌회전 + 전진 |
| 3 Right | `RIGHT` + `UP` | 우회전 + 전진 |
| 4 Space | `SPACE` + `UP` | 스페이스 + 전진 |

**`UP`(전진)은 후진할 때만 빼고 항상 눌려 있다.** 2·3·4 사이를 오갈 때는
`UP` 을 건드리지 않고 나머지 키만 교체하므로 전진이 끊기지 않는다.

### 단계별 요점

| 단계 | 내용 | 비고 |
| --- | --- | --- |
| ① 에포킹 | `get_current_board_data(N)` 로 **최근 N샘플만 조회** | 버퍼를 비우지 않음. 갱신 주기보다 창이 길면 에포크가 겹친다 |
| ② DC 제거 | 채널별 평균을 뺀다 | **필터를 끌 때는 필수.** 안 빼면 전 채널이 DC 값으로 뭉개진다 |
| ③ 노치 | 60 Hz 콘센트 잡음. Q 가 클수록 좁게 깎는다 | Q=30 → –3dB 폭 약 2 Hz |
| ④ 대역통과 | 아래를 올리면 움직임 잡음이 더 빠진다 | 위쪽은 **나이퀴스트(100 Hz) 미만**이어야 함 |
| 필터 적용 | `filtfilt` — 정방향+역방향 2회 | 시간 지연 없음(영위상). 실효 차수는 2배 |
| 창 함수 | **쓰지 않는다** (직사각창) | FFT 를 안 하므로 불필요. 곱하면 오히려 평균을 왜곡 |
| ⑤⑥ 특징 | **Hilbert 포락선 평균**. RMS 아님 | 1차시는 RMS, 여기는 포락선 |
| 특징 벡터 | **4차원**. 시간·주파수 정보 없음 | 동작 구분은 순전히 네 근육의 활성 비율 |
| ⑥ 모델 | Random Forest, 시간순 8:2 분할 | 섞지 않는다 — 이웃 샘플이 양쪽에 들어가면 정확도가 부풀려짐 |
| ⑦ 다수결 | `VOTE_LEN × UPDATE_PERIOD` 만큼 지연 | 안정성 ↔ 반응성 트레이드오프 |
| ⑧ 키 출력 | 판정이 **바뀔 때만** 이전 키를 떼고 새 키를 누름 | 같은 판정이 이어지면 계속 눌린 상태 |

### 알아둘 한계

- **"쉬는 상태" 클래스가 없다.** 힘을 빼도 4개 중 하나로 판정된다.
  → `MIN_CONFIDENCE` 를 올리면 확신 없을 때 키를 떼도록 할 수 있다.
- **전극 위치가 성능을 지배한다.** 특징이 4차원뿐이라 서로 다른 근육을 잡지
  못하면 활성 패턴이 겹쳐 분류가 불가능하다.

---

## 파라미터 조정 가이드

### 짧은 에포크를 쓰려면 — 필터가 걸림돌

`filtfilt` 계열은 경계 왜곡을 줄이려 신호 양 끝에 **패딩**을 붙이는데,
**입력이 그 패딩보다 길어야 한다.** 짧으면 이 에러가 난다.

```
ValueError: The length of the input vector x must be greater than padlen, which is 27.
```

대역통과 차수별 최소 에포크 (200 Hz 기준):

| `FILTER_ORDER` | 필요 padlen | 최소 에포크 |
| --- | --- | --- |
| 4 | 27 | 28샘플 = **0.14초** |
| 3 | 21 | 22샘플 = 0.11초 |
| 2 | 15 | 16샘플 = 0.08초 |

셀 1 이 이 조건을 **미리 검사**해서, 분류 단계가 아니라 그 자리에서 알려준다.

해결책 세 가지:
1. `EPOCH_SEC` 를 늘린다 (신호 품질에도 가장 좋음)
2. `FILTER_ORDER` 를 낮춘다
3. `USE_NOTCH` / `USE_BANDPASS` 를 끈다

### 필터를 끌 때 — `REMOVE_DC` 는 반드시 켤 것

Ganglion 원신호에는 수백 µV 의 **DC 오프셋**이 실려 있다. 대역통과가 켜져 있으면
자동으로 빠지지만, 끄면 그대로 남아 **포락선이 전부 DC 값으로 뭉개진다.**

0.1초 에포크 · 합성 4동작 실측:

| 설정 | 특징 벡터 | 분류 정확도 |
| --- | --- | --- |
| 필터X, DC 유지 | `[492, 502, 506, 505]` ← 구분 불가 | 62.5% |
| 필터X, DC 유지 + DC 변동 | 세션 중 오프셋이 흔들림 | **18.8%** (우연 25%) |
| 필터X, **DC 제거** | `[67.6, 43.4, 39.2, 44.8]` | 93.8% |
| 필터X, **DC 제거** + DC 변동 | | **96.9%** |

DC 를 빼면 오프셋이 흔들려도 문제없다. 안 빼면 **우연보다 못한 성능**이 된다.

> 필터를 모두 끄면 60 Hz 전원 잡음과 움직임 잡음이 특징에 그대로 섞인다.
> 채널마다 유입량이 다르면 분류를 방해하므로, 잘 안 되면 노치부터 다시 켜 볼 것.

### 에포크 길이 (`EPOCH_SEC`) — 반응속도 ↔ 잡음 제거

짧으면 빠르게 반응하지만 필터가 채 안정되기 전에 구간이 끝나 **60 Hz 가 덜 빠진다.**

합성 신호 실측 — 60 Hz 잡음 잔차 (같은 진폭의 통과대역 신호 대비 %):

| `NOTCH_Q` | 0.50초 | 0.75초 | 1.00초 | 2.00초 |
| --- | --- | --- | --- | --- |
| **30** (기본) | **24.2%** | 16.5% | 12.5% | 6.3% |
| 10 | 11.4% | 7.6% | 5.7% | 2.9% |
| 5 | 7.5% | 5.1% | 3.8% | 2.0% |
| 3 | 6.9% | 4.7% | 3.6% | 1.9% |

원인은 노치의 **과도응답**이다. 시정수는 대략 `Q / (π·f0)` 이라
`Q=30` 이면 약 159 ms — 0.5초 에포크는 3 시정수밖에 안 되어 안정 전에 끝난다.
`Q=5` 면 27 ms 로 충분히 안정된다.

> **다만 Q 를 낮추면 그만큼 실제 EMG 도 깎인다.** –3dB 폭이 `f0/Q` 이므로
> `Q=30` 은 59~61 Hz 만, `Q=5` 는 **54~66 Hz 를 통째로** 제거한다.
> 기본값은 원본 실습과 같은 `Q=30` 으로 두었다. 60 Hz 유입이 심하면
> `EPOCH_SEC` 를 늘리는 쪽이 신호 손실 없이 해결하는 방법이다.

이 잔차는 채널마다 비슷하게 실리므로 **전 채널 공통 오프셋**처럼 작용한다.
채널별 60 Hz 유입량이 크게 다를 때(전극 접촉 차이) 분류를 방해한다.

### 그 밖에

| 파라미터 | 올리면 | 내리면 |
| --- | --- | --- |
| `EPOCH_SEC` | 잡음 제거 ↑, 반응 느려짐 | 반응 빠름, 60 Hz 잔차 ↑ |
| `UPDATE_PERIOD` | CPU 부담 ↓ | 갱신 촘촘, 에포크 중첩 ↑ |
| `BAND_LOW` | 움직임 잡음 ↓, 저주파 EMG 손실 | EMG 성분 보존, 드리프트 유입 |
| `BAND_HIGH` | 고주파 EMG 보존 | 잡음 ↓ (**100 Hz 미만 필수**) |
| `VOTE_LEN` | 판정 안정, 지연 ↑ | 반응 빠름, 튀는 판정 ↑ |
| `N_REPEAT` | 학습 데이터 ↑ | 수집 시간 ↓ |
| `MIN_CONFIDENCE` | 애매할 때 키를 뗀다 | 항상 무언가를 누른다 |

---

## 실행 순서

| 셀 | 내용 |
| --- | --- |
| 1 | **파라미터** — 여기만 고친다 |
| 2 | 보드 연결 |
| 3 | 캘리브레이션 (동작 4개 학습 데이터 수집) |
| 4 | 모델 학습 |
| 5 | 실시간 분류 → 방향키 출력 |
| 6 | 비상 키 해제 (키가 눌린 채 남았을 때) |
| 7 | 연결 해제 ⚠️ 꼭 실행 |

> 파라미터를 바꿨으면 **셀 1 을 다시 실행**한 뒤 셀 3(캘리브레이션)부터 다시 한다.
> 필터가 바뀌면 특징 분포가 달라져 기존 모델을 쓸 수 없다.

In [1]:
# ============================================================
# 1. 파라미터  —  여기만 고친다
# ============================================================
import time
import numpy as np
from collections import deque, Counter
from datetime import datetime
from scipy import signal as sp_signal
from sklearn.ensemble import RandomForestClassifier
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds

# ---------------- 연결 ----------------
COM_PORT = 'COM5'          # BLED112 동글 포트

# ---------------- 에포킹 ----------------
EPOCH_SEC       = 0.1      # 한 번에 분석할 구간 길이 (초)
UPDATE_PERIOD   = 0.1      # 몇 초마다 판정할지 (초)
                           #   EPOCH_SEC 보다 짧으면 에포크가 겹친다
                           #   0.1 / 0.1  -> 중첩 없음 (현재)
                           #   0.5 / 0.25 -> 50% 중첩

# ---------------- 필터 ----------------
# 짧은 에포크(0.2초 미만)에서는 filtfilt 가 패딩 길이를 못 채워 에러가 난다.
# 그럴 땐 필터를 끄거나 EPOCH_SEC 를 늘린다. 필요 길이는 아래에서 자동 검사한다.
REMOVE_DC    = True        # DC 오프셋(수백 µV) 제거. 필터를 끌 때는 반드시 True
                           #   끄면 네 채널이 전부 DC 값으로 뭉개져 분류가 안 된다
                           #   (실측: DC 변동 있을 때 정확도 97% -> 19%)

USE_NOTCH    = False       # 60 Hz 전원 잡음 노치
NOTCH_HZ     = 60.0        # 노치 주파수 (한국 60Hz)
NOTCH_Q      = 30.0        # 클수록 좁게 깎는다.
                           #   단, 좁을수록 과도응답이 길어 짧은 에포크에서
                           #   60Hz 를 덜 걷어낸다 (0.5초 기준 Q=30 -> 24% 잔류)
                           #   낮추면 잘 빠지지만 실제 EMG 도 깎인다
                           #   (Q=5 면 54~66Hz 를 통째로 제거)

USE_BANDPASS = False       # 대역통과
BAND_LOW     = 30.0        # 하단 (Hz) - 올리면 움직임 잡음 감소
BAND_HIGH    = 95.0        # 상단 (Hz) - 나이퀴스트(100Hz) 미만이어야 함
FILTER_ORDER = 4           # Butterworth 차수 (filtfilt 라 실효 2배)
                           #   차수가 높을수록 필요한 에포크 길이도 길어진다

# ---------------- 분류 ----------------
GESTURES = {               # 동작 번호 : 이름
    1: 'Reverse',
    2: 'Left',
    3: 'Right',
    4: 'Space',
}
N_REPEAT       = 40        # 동작당 학습 샘플 수 (약 N_REPEAT*UPDATE_PERIOD 초)
N_TREES        = 100       # Random Forest 트리 개수
VOTE_LEN       = 5         # 다수결에 쓸 최근 예측 개수
MIN_CONFIDENCE = 0.0       # 이 확신도 미만이면 키를 뗀다 (0.0 = 끔, 예: 0.5)

# ---------------- 키 출력 ----------------
RUN_SEC           = 60     # 실시간 실행 시간 (초)
KEY_MODE          = 'scan' # 'scan' 또는 'vk' (scan 이 안 먹으면 vk)
REPEAT_WHILE_HELD = True   # 눌린 동안 keydown 재전송 (자동 반복 흉내)

# 키 코드 (scan = 물리 키 위치, vk = 가상 키)
KEY_SCAN = {'UP': 0x48, 'DOWN': 0x50, 'LEFT': 0x4B, 'RIGHT': 0x4D, 'SPACE': 0x39}
KEY_VK   = {'UP': 0x26, 'DOWN': 0x28, 'LEFT': 0x25, 'RIGHT': 0x27, 'SPACE': 0x20}

# 확장키(extended) 여부. 방향키는 True, 일반 키는 False.
# 스페이스에 확장 플래그를 주면 엉뚱한 키로 인식되므로 반드시 구분해야 한다.
KEY_EXT  = {'UP': True, 'DOWN': True, 'LEFT': True, 'RIGHT': True, 'SPACE': False}

KEY_ORDER = ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']   # 누르고 떼는 순서

# 동작 -> 동시에 누를 키 조합. 튜플이므로 몇 개든 넣을 수 있다.
#   UP(전진) 은 후진할 때만 빼고 항상 눌려 있다.
KEY_COMBO = {
    1: ('DOWN',),             # 후진 - UP 을 누르지 않는다
    2: ('LEFT',  'UP'),       # 좌회전 + 전진
    3: ('RIGHT', 'UP'),       # 우회전 + 전진
    4: ('SPACE', 'UP'),       # 스페이스 + 전진
}

# ============================================================
# 아래는 위 값에서 파생된다 - 직접 고칠 필요 없음
# ============================================================
BOARD_ID = BoardIds.GANGLION_BOARD
FS       = BoardShim.get_sampling_rate(BOARD_ID)      # 200 Hz
EMG_CH   = BoardShim.get_eeg_channels(BOARD_ID)       # 생체전위 채널 -> EMG 로 사용
N_CH     = len(EMG_CH)
NYQUIST  = FS / 2

EPOCH_SAMPLES = int(round(FS * EPOCH_SEC))

# 파라미터 검증 - 잘못된 값으로 필터를 설계하면 조용히 이상한 결과가 나온다
assert EPOCH_SAMPLES >= 8, f'에포크가 너무 짧습니다 ({EPOCH_SAMPLES} 샘플). EPOCH_SEC 를 늘리세요'
assert not (USE_BANDPASS and BAND_HIGH >= NYQUIST), \
    f'BAND_HIGH({BAND_HIGH}) 는 나이퀴스트({NYQUIST}) 미만이어야 합니다'
assert not (USE_BANDPASS and not (0 < BAND_LOW < BAND_HIGH)), \
    f'BAND_LOW({BAND_LOW}) < BAND_HIGH({BAND_HIGH}) 여야 합니다'
assert not (USE_NOTCH and NOTCH_HZ >= NYQUIST), \
    f'NOTCH_HZ({NOTCH_HZ}) 는 나이퀴스트({NYQUIST}) 미만이어야 합니다'
assert set(GESTURES) <= set(KEY_COMBO), 'GESTURES 의 번호가 KEY_COMBO 에 없습니다'
for _lbl, _combo in KEY_COMBO.items():
    assert len(_combo) > 0, f'KEY_COMBO[{_lbl}] 가 비어 있습니다'
    assert set(_combo) <= set(KEY_SCAN), \
        f'KEY_COMBO[{_lbl}] 에 모르는 키 이름: {set(_combo) - set(KEY_SCAN)}'
assert set(KEY_SCAN) == set(KEY_VK) == set(KEY_EXT) == set(KEY_ORDER), \
    'KEY_SCAN / KEY_VK / KEY_EXT / KEY_ORDER 의 키 목록이 서로 다릅니다'

# 화면 표시용 이름 (예: 'LEFT+UP')
KEY_LABEL = {k: '+'.join(v) for k, v in KEY_COMBO.items()}

# 필터 계수 (한 번만 설계) + 에포크 길이 검사
#
# filtfilt 계열은 경계 왜곡을 줄이려 신호 양 끝에 패딩을 붙이는데,
# 입력이 그 패딩보다 길어야 한다. 짧으면 다음 에러가 난다.
#   ValueError: The length of the input vector x must be greater than padlen
# 분류 단계에서 터지지 않도록 여기서 미리 확인한다.
_need = []

if USE_NOTCH:
    b_notch, a_notch = sp_signal.iirnotch(NOTCH_HZ, NOTCH_Q, fs=FS)
    _need.append(('노치', 3 * max(len(b_notch), len(a_notch))))
else:
    b_notch = a_notch = None

if USE_BANDPASS:
    sos_band = sp_signal.butter(FILTER_ORDER, [BAND_LOW, BAND_HIGH],
                                btype='band', fs=FS, output='sos')
    _ns  = sos_band.shape[0]
    _pad = 3 * (2 * _ns + 1 - min((sos_band[:, 2] == 0).sum(),
                                  (sos_band[:, 5] == 0).sum()))
    _need.append((f'대역통과 {FILTER_ORDER}차', _pad))
else:
    sos_band = None

for _name, _pad in _need:
    assert EPOCH_SAMPLES > _pad, (
        f'{_name} 필터는 {_pad + 1}샘플({(_pad + 1) / FS:.3f}초) 이상이 필요한데 '
        f'에포크가 {EPOCH_SAMPLES}샘플({EPOCH_SEC}초)뿐입니다.\n'
        f'      해결: EPOCH_SEC 를 늘리거나, FILTER_ORDER 를 낮추거나, '
        f'해당 필터를 끄세요(USE_NOTCH / USE_BANDPASS).')


def extract_features(epoch):
    """에포크 -> 특징 벡터.

    epoch : (샘플, 채널) 원시 EMG
    반환  : (채널,) 채널별 포락선 평균

    DC제거 -> 노치 -> 대역통과 -> Hilbert 포락선 -> 구간 평균.
    각 필터는 켜져 있을 때만 적용된다.
    filtfilt / sosfiltfilt 는 정방향+역방향 2회 통과라 위상 지연이 없다.
    """
    x = epoch
    if REMOVE_DC:
        # DC 오프셋을 안 빼면 포락선이 전부 DC 값으로 뭉개진다.
        # 대역통과를 켜면 어차피 DC 가 빠지지만, 껐을 때는 이게 유일한 방어선.
        x = x - x.mean(axis=0)
    if USE_NOTCH:
        x = sp_signal.filtfilt(b_notch, a_notch, x, axis=0)    # 노치는 (b,a) 형식
    if USE_BANDPASS:
        x = sp_signal.sosfiltfilt(sos_band, x, axis=0)         # 대역통과는 sos 형식
    envelope = np.abs(sp_signal.hilbert(x, axis=0))            # 순간 진폭 = 포락선
    return envelope.mean(axis=0)                               # 채널당 숫자 1개


print('파라미터 확인')
print('=' * 60)
print(f'  포트        : {COM_PORT}')
print(f'  샘플링      : {FS} Hz, 채널 {N_CH}개 {EMG_CH}, 나이퀴스트 {NYQUIST:.0f} Hz')
print(f'  에포크      : {EPOCH_SEC}초 = {EPOCH_SAMPLES} 샘플')
print(f'  갱신 주기   : {UPDATE_PERIOD}초', end='')
if UPDATE_PERIOD < EPOCH_SEC:
    print(f'  (중첩 {100*(1-UPDATE_PERIOD/EPOCH_SEC):.0f}%)')
else:
    print('  (중첩 없음)')
print(f'  DC 제거     : {"켬" if REMOVE_DC else "끔"}')
if USE_NOTCH:
    print(f'  노치        : {NOTCH_HZ:.0f} Hz, Q={NOTCH_Q:.0f}')
else:
    print('  노치        : 끔')
if USE_BANDPASS:
    print(f'  대역통과    : {BAND_LOW:.0f}-{BAND_HIGH:.0f} Hz, Butterworth {FILTER_ORDER}차'
          f' (filtfilt -> 실효 {FILTER_ORDER*2}차)')
else:
    print('  대역통과    : 끔')
if not (USE_NOTCH or USE_BANDPASS):
    print('                (필터 없음 - 원신호 진폭만 본다.'
          ' 60Hz 잡음과 움직임 잡음이 그대로 들어온다)')
    if not REMOVE_DC:
        print('                [!] REMOVE_DC 가 꺼져 있습니다. 분류가 거의 불가능합니다')
print(f'  특징        : 채널별 Hilbert 포락선 평균 -> {N_CH}차원')
print(f'  모델        : Random Forest, 트리 {N_TREES}개')
print(f'  다수결      : 최근 {VOTE_LEN}회 -> 지연 약 {VOTE_LEN*UPDATE_PERIOD:.2f}초')
print(f'  확신도 하한 : ' + (f'{MIN_CONFIDENCE*100:.0f}%' if MIN_CONFIDENCE > 0 else '끔'))
print(f'  학습 데이터 : {len(GESTURES)}동작 x {N_REPEAT}회 = {len(GESTURES)*N_REPEAT}개'
      f' (동작당 약 {N_REPEAT*UPDATE_PERIOD:.0f}초)')
print('  동작 -> 키')
for k in sorted(GESTURES):
    combo = KEY_COMBO[k]
    tag = '  (동시 입력)' if len(combo) > 1 else ''
    print(f'      [{k}] {GESTURES[k]:<8} -> {KEY_LABEL[k]}{tag}')
print('=' * 60)


파라미터 확인
  포트        : COM5
  샘플링      : 200 Hz, 채널 4개 [1, 2, 3, 4], 나이퀴스트 100 Hz
  에포크      : 0.1초 = 20 샘플
  갱신 주기   : 0.1초  (중첩 없음)
  DC 제거     : 켬
  노치        : 끔
  대역통과    : 끔
                (필터 없음 - 원신호 진폭만 본다. 60Hz 잡음과 움직임 잡음이 그대로 들어온다)
  특징        : 채널별 Hilbert 포락선 평균 -> 4차원
  모델        : Random Forest, 트리 100개
  다수결      : 최근 5회 -> 지연 약 0.50초
  확신도 하한 : 끔
  학습 데이터 : 4동작 x 40회 = 160개 (동작당 약 4초)
  동작 -> 키
      [1] Reverse  -> DOWN
      [2] Left     -> LEFT+UP  (동시 입력)
      [3] Right    -> RIGHT+UP  (동시 입력)
      [4] Space    -> SPACE+UP  (동시 입력)


In [2]:
# ============================================================
# 2. 보드 연결
# ============================================================
BoardShim.disable_board_logger()

params = BrainFlowInputParams()
params.serial_port = COM_PORT
params.timeout     = 40

board     = BoardShim(BOARD_ID, params)
connected = False

print(f'{COM_PORT} 로 Ganglion 연결 시도... (첫 시도는 40초까지 걸릴 수 있음)')
for attempt in range(1, 4):
    t0 = time.time()
    try:
        board.prepare_session()
        connected = True
        print(f'[{attempt}/3] 연결 성공 ({time.time()-t0:.1f}초)')
        break
    except Exception as e:
        print(f'[{attempt}/3] 실패 ({time.time()-t0:.1f}초): {e}')
        try:
            board.release_session()
        except Exception:
            pass
        if attempt < 3:
            time.sleep(3)

if connected:
    board.start_stream(450000, '')
    print('스트리밍 시작')
else:
    print()
    print('연결 실패. 확인할 것:')
    print('  - OpenBCI GUI 종료 (작업관리자에서 javaw 까지)')
    print('  - 다른 노트북 커널이 COM 포트를 잡고 있는지 (셀 7 로 해제)')
    print('  - 보드 전원 ON, 배터리 확인')
    print('  - brainflow 가 5.20.0 인지 (5.21+ 는 BLED112 연결 불가)')


COM5 로 Ganglion 연결 시도... (첫 시도는 40초까지 걸릴 수 있음)
[1/3] 연결 성공 (1.5초)
스트리밍 시작


---
## 3 · 캘리브레이션

동작마다 **3·2·1 카운트다운 후 자세를 유지**한 채로 수집한다.
동작당 `N_REPEAT` 개, 4동작이면 `4 × N_REPEAT` 개의 특징 벡터가 모인다.

> 수집 중에는 자세와 **힘의 세기를 일정하게** 유지할 것. 흔들리면 모델이 헷갈린다.

끝나면 동작별 채널 평균 표가 나온다. **동작끼리 패턴이 뚜렷하게 달라야** 분류가 된다.
비슷하면 전극을 서로 다른 근육으로 옮기고 다시 수집한다.

In [6]:
# ============================================================
# 3. 캘리브레이션  —  동작마다 자세를 유지한 채로 수집
# ============================================================
if not connected:
    print('보드가 연결되지 않았습니다. 셀 2 를 먼저 성공시키세요.')
else:
    def collect_gesture(label, name, n_repeat):
        print(f'\n>>> [{label}] {name} 준비')
        for c in (3, 2, 1):
            print(f'    {c}...')
            time.sleep(1)
        print(f'    시작! {name} 자세를 유지하세요')

        feats = np.zeros((n_repeat, N_CH))
        for i in range(n_repeat):
            d = board.get_current_board_data(EPOCH_SAMPLES)
            feats[i] = extract_features(d[EMG_CH, :].T)     # (샘플, 채널) 로 전치
            time.sleep(UPDATE_PERIOD)

            pct = (i + 1) / n_repeat
            bar = '#' * int(30 * pct) + '.' * (30 - int(30 * pct))
            print(f'\r    [{bar}] {pct*100:3.0f}%  ({i+1}/{n_repeat})', end='', flush=True)
        print()
        return feats

    board.get_board_data()          # 버퍼 비우기

    X_list, Y_list = [], []
    for label in sorted(GESTURES):
        feats = collect_gesture(label, GESTURES[label], N_REPEAT)
        X_list.append(feats)
        Y_list.append(np.full(N_REPEAT, label))

    X = np.vstack(X_list)
    Y = np.concatenate(Y_list)

    print()
    print('=' * 60)
    print(f'캘리브레이션 완료 : X={X.shape}  Y={Y.shape}')
    print('=' * 60)
    print(f'{"동작":<12}' + ''.join(f'{f"Ch{i+1}":>10}' for i in range(N_CH)))
    print('-' * 60)
    for label in sorted(GESTURES):
        m = X[Y == label].mean(axis=0)
        print(f'{GESTURES[label]:<12}' + ''.join(f'{v:>10.1f}' for v in m))
    print('-' * 60)
    print('동작끼리 채널 패턴이 뚜렷하게 달라야 분류가 된다.')
    print('비슷하면 전극을 서로 다른 근육으로 옮기고 이 셀을 다시 실행할 것.')



>>> [1] Reverse 준비
    3...
    2...
    1...
    시작! Reverse 자세를 유지하세요
    [##############################] 100%  (40/40)

>>> [2] Left 준비
    3...
    2...
    1...
    시작! Left 자세를 유지하세요
    [##############################] 100%  (40/40)

>>> [3] Right 준비
    3...
    2...
    1...
    시작! Right 자세를 유지하세요
    [##############################] 100%  (40/40)

>>> [4] Space 준비
    3...
    2...
    1...
    시작! Space 자세를 유지하세요
    [##############################] 100%  (40/40)

캘리브레이션 완료 : X=(160, 4)  Y=(160,)
동작                 Ch1       Ch2       Ch3       Ch4
------------------------------------------------------------
Reverse         1156.5    1423.5     906.9     947.8
Left             355.2     496.8     555.9     538.9
Right            777.3     605.2     555.0     420.5
Space            479.3     470.6     575.0     581.0
------------------------------------------------------------
동작끼리 채널 패턴이 뚜렷하게 달라야 분류가 된다.
비슷하면 전극을 서로 다른 근육으로 옮기고 이 셀을 다시 실행할 것.


In [7]:
# ============================================================
# 4. 모델 학습  (시간순 8:2 분할 - 섞지 않는다)
# ============================================================
if 'X' not in dir():
    print('먼저 셀 3 으로 데이터를 수집하세요.')
else:
    # 무작위로 섞으면 0.25초 간격 이웃 샘플이 학습셋과 시험셋에 나뉘어 들어가
    # 정확도가 부풀려진다. 동작별로 앞 80% 학습 / 뒤 20% 시험.
    train_idx, test_idx = [], []
    for label in sorted(GESTURES):
        idx = np.where(Y == label)[0]
        cut = int(len(idx) * 0.8)
        train_idx.extend(idx[:cut])
        test_idx.extend(idx[cut:])
    train_idx, test_idx = np.array(train_idx), np.array(test_idx)

    clf = RandomForestClassifier(n_estimators=N_TREES, random_state=42)
    clf.fit(X[train_idx], Y[train_idx])

    Y_pred   = clf.predict(X[test_idx])
    accuracy = (Y_pred == Y[test_idx]).mean() * 100

    print(f'학습 {len(train_idx)}개 / 시험 {len(test_idx)}개')
    print(f'시험 정확도 : {accuracy:.1f}%')
    print()

    print('동작별 적중')
    print('-' * 40)
    for label in sorted(GESTURES):
        sel = Y[test_idx] == label
        if sel.any():
            hit = (Y_pred[sel] == label).sum()
            print(f'  [{label}] {GESTURES[label]:<10} {hit}/{sel.sum()}')
    print('-' * 40)

    print()
    print('채널 중요도 (0 에 가까우면 그 전극은 판별에 도움이 안 된다)')
    for i, imp in enumerate(clf.feature_importances_):
        print(f'  Ch{i+1} {"#" * int(imp * 40):<40} {imp*100:4.1f}%')


학습 128개 / 시험 32개
시험 정확도 : 68.8%

동작별 적중
----------------------------------------
  [1] Reverse    5/8
  [2] Left       4/8
  [3] Right      6/8
  [4] Space      7/8
----------------------------------------

채널 중요도 (0 에 가까우면 그 전극은 판별에 도움이 안 된다)
  Ch1 #############                            34.1%
  Ch2 #########                                23.6%
  Ch3 ######                                   17.2%
  Ch4 ##########                               25.1%


---
## 5 · 실시간 분류 → 방향키

판정이 **바뀔 때만**, 그리고 **바뀐 키만** 갈아끼운다. 같은 판정이 이어지면 계속 눌린 상태.

| 동작 | 누르는 키 |
| --- | --- |
| 1 Reverse | `DOWN` (UP 없음) |
| 2 Left | `LEFT` + `UP` |
| 3 Right | `RIGHT` + `UP` |
| 4 Space | `SPACE` + `UP` |

```
판정      2          2        3            4             1
       LEFT+UP    (유지)   RIGHT+UP     SPACE+UP        DOWN
전송  ↓UP ↓LEFT     -    ↑LEFT ↓RIGHT  ↑RIGHT ↓SPACE  ↑UP ↑SPACE ↓DOWN
                     └──── UP 은 계속 눌린 채 유지 ────┘  └ 후진이라 UP 을 뗌
```

**2·3·4 사이를 오갈 때 `UP` 은 떼지 않는다.** 떼었다 다시 누르면 그 순간 전진이
끊겨 덜컥거린다. 집합 차집합으로 필요한 키만 교체한다.
후진(1)으로 갈 때만 `UP` 이 떨어진다.

조합을 바꾸려면 셀 1 의 `KEY_COMBO` 만 고치면 된다. 키는 몇 개든 넣을 수 있다.

```python
KEY_COMBO = {
    1: ('DOWN',),
    2: ('LEFT',  'UP'),
    3: ('RIGHT', 'UP'),
    4: ('SPACE', 'UP'),
}
```

쓸 수 있는 키 이름은 `KEY_SCAN` 에 정의된 것들이다 (`UP` `DOWN` `LEFT` `RIGHT` `SPACE`).
다른 키를 추가하려면 `KEY_SCAN` · `KEY_VK` · `KEY_EXT` · `KEY_ORDER` 네 곳에 같이 넣는다.
`KEY_EXT` 는 확장키 여부로, **방향키만 `True`** 다.

### 사용법

1. 방향키가 먹는 창을 열어 둔다 (게임, 브라우저, 메모장 등)
2. 이 셀 실행 → **5초 카운트다운 동안 그 창을 클릭**
3. 동작을 취하면 방향키가 눌린다
4. 중단은 **ESC**

> ⚠️ 포커스를 가진 **아무 창에나** 입력된다. 실행 중 이 노트북을 클릭하지 말 것.
> 표시되는 `포커스: ...` 가 대상 창인지 확인한다.

### 자동 반복에 대해

Windows 는 **주입된 키에 자동 반복을 만들어 주지 않는다** (자동 반복은 실제 키보드
드라이버가 생성). 그래서:

- **게임·`GetAsyncKeyState` 방식 앱** → keydown 한 번으로 "눌림" 유지, 정상 동작
- **메모장처럼 `WM_KEYDOWN` 반복에 의존하는 앱** → 한 칸만 움직이고 멈춤

`REPEAT_WHILE_HELD = True` 가 keyup 없이 keydown 을 매 주기 재전송해 이를 흉내낸다.
게임류가 대상이면 `False` 로 둬도 된다.

### 안전장치

| 장치 | 내용 |
| --- | --- |
| `try/finally` | 예외·ESC·시간만료 어느 쪽이든 키를 뗀다 |
| ESC | 매 주기 확인. Ctrl+C 가 노트북에 안 닿으므로 이게 탈출구 |
| `RUN_SEC` | 자동 종료 |
| 셀 6 | 강제 중단으로 `finally` 가 안 돌았을 때 수동 해제 |
| 포커스 표시 | 키가 어디로 가는지 실시간 확인 |

In [8]:
# ============================================================
# 5. 실시간 분류 -> 방향키 출력
#    판정이 바뀔 때만 이전 키를 떼고 새 키를 누른다 (누름 유지)
# ============================================================
import ctypes
from ctypes import wintypes

user32 = ctypes.WinDLL('user32', use_last_error=True)

INPUT_KEYBOARD        = 1
KEYEVENTF_EXTENDEDKEY = 0x0001
KEYEVENTF_KEYUP       = 0x0002
KEYEVENTF_SCANCODE    = 0x0008
VK_ESCAPE             = 0x1B

KEYMAP = KEY_SCAN if KEY_MODE == 'scan' else KEY_VK

ULONG_PTR = ctypes.c_ulonglong if ctypes.sizeof(ctypes.c_void_p) == 8 else ctypes.c_ulong


# INPUT 의 union 은 MOUSEINPUT / KEYBDINPUT / HARDWAREINPUT 중 가장 큰 것 크기다.
# x64 에서 MOUSEINPUT 이 32바이트라 union 32, INPUT 40 이 된다.
# 이 크기를 틀리게 잡으면 SendInput 이 ERROR_INVALID_PARAMETER(87) 로 조용히 실패한다.
class MOUSEINPUT(ctypes.Structure):
    _fields_ = [('dx', wintypes.LONG), ('dy', wintypes.LONG),
                ('mouseData', wintypes.DWORD), ('dwFlags', wintypes.DWORD),
                ('time', wintypes.DWORD), ('dwExtraInfo', ULONG_PTR)]


class KEYBDINPUT(ctypes.Structure):
    _fields_ = [('wVk', wintypes.WORD), ('wScan', wintypes.WORD),
                ('dwFlags', wintypes.DWORD), ('time', wintypes.DWORD),
                ('dwExtraInfo', ULONG_PTR)]


class HARDWAREINPUT(ctypes.Structure):
    _fields_ = [('uMsg', wintypes.DWORD),
                ('wParamL', wintypes.WORD), ('wParamH', wintypes.WORD)]


class _INPUTunion(ctypes.Union):
    _fields_ = [('mi', MOUSEINPUT), ('ki', KEYBDINPUT), ('hi', HARDWAREINPUT)]


class INPUT(ctypes.Structure):
    _fields_ = [('type', wintypes.DWORD), ('u', _INPUTunion)]


# argtypes 를 지정하지 않으면 64비트에서 포인터가 잘려 역시 실패한다.
user32.SendInput.argtypes = (wintypes.UINT, ctypes.POINTER(INPUT), ctypes.c_int)
user32.SendInput.restype  = wintypes.UINT
user32.GetForegroundWindow.restype = wintypes.HWND
user32.GetWindowTextW.argtypes = (wintypes.HWND, wintypes.LPWSTR, ctypes.c_int)

_EXPECT = 40 if ctypes.sizeof(ctypes.c_void_p) == 8 else 28
if ctypes.sizeof(INPUT) != _EXPECT:
    raise SystemExit(f'sizeof(INPUT)={ctypes.sizeof(INPUT)} (기대값 {_EXPECT}). '
                     'SendInput 이 87 에러로 실패합니다.')


def _send(name, up):
    """키 하나에 keydown/keyup 을 보낸다.

    방향키는 확장키(extended)라 EXTENDEDKEY 플래그가 필요하지만,
    스페이스 같은 일반 키에 이 플래그를 주면 엉뚱한 키로 인식된다.
    그래서 KEY_EXT 로 키마다 구분한다.
    """
    flags = KEYEVENTF_EXTENDEDKEY if KEY_EXT[name] else 0
    if up:
        flags |= KEYEVENTF_KEYUP
    code = KEYMAP[name]
    if KEY_MODE == 'scan':
        ki = KEYBDINPUT(0, code, flags | KEYEVENTF_SCANCODE, 0, 0)
    else:
        ki = KEYBDINPUT(code, 0, flags, 0, 0)
    inp = INPUT()
    inp.type = INPUT_KEYBOARD
    inp.u.ki = ki
    arr = (INPUT * 1)(inp)
    if user32.SendInput(1, arr, ctypes.sizeof(INPUT)) != 1:
        print(f'\n[!] SendInput 실패 ({name}) GetLastError={ctypes.get_last_error()}')
        return False
    return True


# held_keys = 지금 눌려 있는 키 이름 집합.  held = 그 조합에 해당하는 동작 번호
held_keys = set()
held      = None


def hold_action(n):
    """동작 n 의 키 조합을 누른 상태로 만든다.

    필요한 것만 갈아끼운다. 예를 들어 LEFT+UP -> RIGHT+UP 으로 바뀌면
    UP 은 건드리지 않고 LEFT 만 떼고 RIGHT 를 누른다.
    (UP 을 떼었다 다시 누르면 그 순간 입력이 끊겨 움직임이 덜컥거린다)
    """
    global held_keys, held
    want = set(KEY_COMBO[n])

    if want == held_keys:
        # 주입된 키에는 OS 가 자동 반복을 만들어 주지 않는다.
        # 반복 입력이 필요한 앱을 위해 keydown 을 다시 보낸다.
        if REPEAT_WHILE_HELD:
            for k in KEY_ORDER:
                if k in want:
                    _send(k, False)
        held = n
        return True

    ok = True
    for k in KEY_ORDER:                 # 먼저 필요 없어진 키를 뗀다
        if k in held_keys - want:
            ok &= _send(k, True)
    for k in KEY_ORDER:                 # 그 다음 새 키를 누른다
        if k in want - held_keys:
            ok &= _send(k, False)

    held_keys = set(want)
    held      = n
    return ok


def release_held():
    """눌러 둔 키를 모두 뗀다."""
    global held_keys, held
    for k in KEY_ORDER:
        if k in held_keys:
            _send(k, True)
    held_keys = set()
    held      = None


def release_all_keys():
    """비상용 - 상태와 무관하게 모든 키에 keyup 을 보낸다."""
    global held_keys, held
    for k in KEY_ORDER:
        _send(k, True)
    held_keys = set()
    held      = None
    print('모든 키를 뗐습니다.')


def foreground_title():
    """포커스를 가진 창 제목 - 키가 어디로 가는지 확인용"""
    buf = ctypes.create_unicode_buffer(256)
    user32.GetWindowTextW(user32.GetForegroundWindow(), buf, 256)
    return buf.value or '(제목 없음)'


def esc_pressed():
    return bool(user32.GetAsyncKeyState(VK_ESCAPE) & 0x8000)


# ---- 실행 ----
if 'clf' not in dir():
    print('먼저 셀 4 로 모델을 학습하세요.')
elif not connected:
    print('보드가 연결되지 않았습니다.')
else:
    print('=' * 68)
    print('실시간 분류 -> 방향키 출력')
    print('=' * 68)
    print(f'  판정 주기 : {UPDATE_PERIOD}초   총 {RUN_SEC}초')
    print(f'  입력 방식 : {KEY_MODE}, 자동반복 {"켬" if REPEAT_WHILE_HELD else "끔"}')
    print('  동작 -> 키: ' + ' / '.join(f'{GESTURES[k]}->{KEY_LABEL[k]}'
                                        for k in sorted(GESTURES)))
    print('  중단      : ESC (키도 자동으로 뗀다)')
    print('=' * 68)
    for c in range(5, 0, -1):
        print(f'  {c}초 뒤 시작 - 지금 대상 창을 클릭하세요!')
        time.sleep(1)

    recent  = deque(maxlen=VOTE_LEN)
    changes = []
    fails   = 0
    aborted = False
    board.get_board_data()

    t_end = time.time() + RUN_SEC

    # 어떤 이유로 끝나든 키는 반드시 뗀다. 없으면 키가 눌린 채 남는다.
    try:
        while time.time() < t_end:
            t_cycle = time.time()

            if esc_pressed():
                aborted = True
                break

            d = board.get_current_board_data(EPOCH_SAMPLES)
            if d.shape[1] < EPOCH_SAMPLES:
                time.sleep(0.05)
                continue

            feat   = extract_features(d[EMG_CH, :].T).reshape(1, -1)
            proba  = clf.predict_proba(feat)[0]
            pred   = int(clf.classes_[proba.argmax()])
            conf   = proba.max()
            recent.append(pred)
            stable = Counter(recent).most_common(1)[0][0]

            prev = frozenset(held_keys)
            if conf < MIN_CONFIDENCE:
                release_held()                  # 확신 없으면 아무 키도 안 누름
            elif not hold_action(stable):
                fails += 1
            if frozenset(held_keys) != prev:
                changes.append((datetime.now().strftime('%H:%M:%S'), prev,
                                frozenset(held_keys)))

            remain = t_end - time.time()
            keys   = '+'.join(k for k in KEY_ORDER if k in held_keys)
            state  = f'[누름] {keys}' if held_keys else '[뗌]'
            print(f'\r[{remain:5.1f}s] [{stable}] {GESTURES[stable]:<7} '
                  f'{conf*100:3.0f}%  {state:20s} | 전환 {len(changes):3d}회 '
                  f'| 포커스: {foreground_title()[:22]:22s}', end='', flush=True)

            dt = UPDATE_PERIOD - (time.time() - t_cycle)
            if dt > 0:
                time.sleep(dt)
    finally:
        release_held()

    print()
    print()
    print('중단됨 (ESC)' if aborted else '종료 (시간 만료)')
    print(f'눌린 키를 모두 뗐습니다. 키 전환 {len(changes)}회, 전송 실패 {fails}회')
    if changes:
        print()
        print('전환 이력 (마지막 12회):')
        for t, a, b in changes[-12:]:
            fr = '+'.join(k for k in KEY_ORDER if k in a) or '(없음)'
            to = '+'.join(k for k in KEY_ORDER if k in b) or '(뗌)'
            print(f'  {t}  {fr:<14} -> {to}')


실시간 분류 -> 방향키 출력
  판정 주기 : 0.1초   총 60초
  입력 방식 : scan, 자동반복 켬
  동작 -> 키: Reverse->DOWN / Left->LEFT+UP / Right->RIGHT+UP / Space->SPACE+UP
  중단      : ESC (키도 자동으로 뗀다)
  5초 뒤 시작 - 지금 대상 창을 클릭하세요!
  4초 뒤 시작 - 지금 대상 창을 클릭하세요!
  3초 뒤 시작 - 지금 대상 창을 클릭하세요!
  2초 뒤 시작 - 지금 대상 창을 클릭하세요!
  1초 뒤 시작 - 지금 대상 창을 클릭하세요!
[ -0.0s] [3] Right    54%  [누름] UP+RIGHT        | 전환 146회 | 포커스: Mario Kart PC - Chrome

종료 (시간 만료)
눌린 키를 모두 뗐습니다. 키 전환 146회, 전송 실패 0회

전환 이력 (마지막 12회):
  14:55:34  UP+RIGHT       -> DOWN
  14:55:34  DOWN           -> UP+RIGHT
  14:55:37  UP+RIGHT       -> UP+SPACE
  14:55:37  UP+SPACE       -> UP+LEFT
  14:55:37  UP+LEFT        -> UP+RIGHT
  14:55:39  UP+RIGHT       -> UP+LEFT
  14:55:40  UP+LEFT        -> UP+SPACE
  14:55:40  UP+SPACE       -> DOWN
  14:55:41  DOWN           -> UP+SPACE
  14:55:41  UP+SPACE       -> DOWN
  14:55:41  DOWN           -> UP+SPACE
  14:55:41  UP+SPACE       -> UP+RIGHT


In [ ]:
# ============================================================
# 6. 비상 키 해제  —  셀 5 를 강제 중단해 키가 눌린 채 남았을 때
# ============================================================
release_all_keys()


In [ ]:
# ============================================================
# 7. 연결 해제   꼭 실행할 것
# ============================================================
try:
    if board.is_prepared():
        board.release_session()
        print('연결 해제 완료. 동글이 사용 가능한 상태입니다.')
    else:
        print('이미 해제되어 있습니다.')
except Exception as e:
    print(f'해제 중 문제: {e}')
    print('커널을 재시작하면 정리됩니다.')
